# Aula 03 · Vídeo 3 — RAG com LangGraph (PT‑BR, Machado de Assis)

Nesta versão, usamos **conteúdo em português** (Machado de Assis, domínio público – Project Gutenberg) para tornar a comparação **Sem RAG vs Com RAG** realista.

### O que este notebook faz
- Monta um índice (FAISS) com um mini‑corpus local **e** um livro público de Machado de Assis.
- Implementa o fluxo em LangGraph: `retrieve → generate_rag` e, em paralelo, uma resposta **Sem RAG** (baseline).
- Mostra **fontes** e **chunks** usados na resposta com RAG.

> Observação: os arquivos do Project Gutenberg estão em domínio público. O download ocorre em tempo de execução do notebook no seu ambiente.


## 1) Setup
- Se `OPENAI_API_KEY` estiver definida, usamos `ChatOpenAI` e `OpenAIEmbeddings`.
- Caso contrário, usamos `FakeEmbeddings` e respostas mock (útil para gravação/offline).

In [13]:
import os, requests, pathlib
from typing import List, Dict
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
USE_OPENAI = bool(OPENAI_API_KEY)
USE_OPENAI

True

## 2) LLM e embeddings
Preferimos OpenAI quando disponível; caso contrário, `FakeEmbeddings`.

In [14]:
try:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
except Exception:
    ChatOpenAI = None
    OpenAIEmbeddings = None

from langchain_core.embeddings import FakeEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

if USE_OPENAI and ChatOpenAI and OpenAIEmbeddings:
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    embeddings = OpenAIEmbeddings()
else:
    llm = None
    embeddings = FakeEmbeddings(size=768)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x76763ebcbf50>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x76763ebcb9d0>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

## 3) Corpus local + **Machado de Assis** (Project Gutenberg, PT‑BR)
Tentamos baixar primeiro **Memórias Póstumas de Brás Cubas** (PG #5415). Se falhar, tentamos **Dom Casmurro** (PG #55752).

In [15]:
DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)

# Mini-corpus local em PT-BR
local_corpus = [
    "RAG combina recuperação de contexto com geração para respostas ancoradas em fontes.",
    "LangGraph estrutura fluxos de IA como grafos de execução com nós e arestas.",
    "Agentes especializados podem ser organizados em grafo para compor soluções complexas.",
]
(DATA_DIR/"local_corpus.txt").write_text("\n".join(local_corpus), encoding="utf-8")

# URLs de livros em PT-BR no Gutenberg (texto simples)
machado_urls = [
    "https://www.gutenberg.org/files/5415/5415-0.txt",   # Memórias Póstumas de Brás Cubas
    "https://www.gutenberg.org/files/55752/55752-0.txt", # Dom Casmurro
]
machado_path = DATA_DIR/"machado.txt"

baixou = False
for url in machado_urls:
    try:
        r = requests.get(url, timeout=30)
        if r.ok and len(r.text) > 5000:
            machado_path.write_text(r.text, encoding="utf-8")
            baixou = True
            print("Baixado:", url)
            break
    except Exception as e:
        print("Falha ao baixar de", url, "|", e)

if not baixou:
    # Conteúdo mínimo de fallback (para demo offline)
    machado_path.write_text(
        "Capítulo I — Do livro: Memórias Póstumas de Brás Cubas (resumo curto OFFLINE em PT-BR).\n\n"
        "... Narração em primeira pessoa, reflexão irônica sobre a vida, sociedade e memórias.\n",
        encoding="utf-8",
    )

len(machado_path.read_text(encoding="utf-8"))

Baixado: https://www.gutenberg.org/files/55752/55752-0.txt


381556

## 4) Chunking e índice (FAISS)
Transformamos o corpus em **chunks** e indexamos para recuperação semântica.

In [16]:
try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.schema import Document

def to_docs(text: str, source: str, chunk_size=900, overlap=150) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    return [Document(page_content=c, metadata={"source": source}) for c in splitter.split_text(text)]

docs: List[Document] = []
docs += to_docs((DATA_DIR/"local_corpus.txt").read_text(encoding="utf-8"), source="local_corpus.txt")
docs += to_docs((DATA_DIR/"machado.txt").read_text(encoding="utf-8"), source="machado.txt")

from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
len(docs)

606

## 5) Estado e nós (LangGraph)
Fluxo: `retrieve` → `generate_rag`. Em paralelo, geramos uma resposta **Sem RAG** para comparação.
Também exibimos **chunks** e **fontes** usados na resposta com RAG.

In [17]:
class RAGState(TypedDict, total=False):
    query: str
    docs: List[Dict]
    resposta_baseline: str
    resposta_rag: str
    fontes: List[str]

def retrieve_node(s: RAGState) -> RAGState:
    query = s.get("query", "")
    # API nova: invoke()
    hits = retriever.invoke(query)
    docs_slim = [{"text": d.page_content, "source": d.metadata.get("source", "")} for d in hits]
    return {"docs": docs_slim, "fontes": list({d["source"] for d in docs_slim})}

def build_prompt(query: str, docs: List[Dict]) -> str:
    contexto = "\n\n".join([f"Fonte: {d['source']}\nTrecho:\n{d['text']}" for d in docs])
    instrucao = (
        "Responda em PT-BR de forma objetiva. Use APENAS o contexto abaixo.\n"
        "Se não houver contexto suficiente, diga que não há informação.\n\n"
        f"Pergunta: {query}\n\nContexto:\n{contexto}\n\nResposta concisa:"
    )
    return instrucao

def baseline_generate_node(s: RAGState) -> RAGState:
    query = s.get("query", "")
    if llm is None:
        return {"resposta_baseline": "(offline) Sem RAG e sem LLM: difícil responder com precisão sobre literatura."}
    else:
        from langchain_core.prompts import PromptTemplate
        prompt = PromptTemplate.from_template(
            "Responda em PT-BR de forma objetiva à pergunta a seguir usando apenas seu conhecimento geral.\nPergunta: {q}\n\nResposta concisa:"
        )
        chain = prompt | llm
        out = chain.invoke({"q": query}).content
        return {"resposta_baseline": out}

def generate_rag_node(s: RAGState) -> RAGState:
    query = s.get("query", "")
    docs = s.get("docs", [])
    if llm is None:
        if not docs:
            return {"resposta_rag": "(offline) Não há contexto suficiente.", "fontes": []}
        snippet = docs[0]["text"][:240].replace("\n", " ")
        return {"resposta_rag": f"(offline) Baseado no contexto: {snippet}...", "fontes": list({d['source'] for d in docs})}
    else:
        from langchain_core.prompts import PromptTemplate
        prompt = PromptTemplate.from_template("{instrucao}")
        chain = prompt | llm
        instrucao = build_prompt(query, docs)
        out = chain.invoke({"instrucao": instrucao}).content
        return {"resposta_rag": out, "fontes": list({d['source'] for d in docs})}


## 6) Grafo e execução (PT‑BR)
Perguntas em **português** para evidenciar diferenças entre **Sem RAG** e **Com RAG** usando Machado de Assis.

In [18]:
g = StateGraph(RAGState)
g.add_node("retrieve", retrieve_node)
g.add_node("generate_rag", generate_rag_node)
g.set_entry_point("retrieve")
g.add_edge("retrieve", "generate_rag")
g.add_edge("generate_rag", END)
app = g.compile()
print(app.get_graph().draw_ascii())

def show_chunks(docs: List[Dict], max_chars=220):
    for i, d in enumerate(docs, 1):
        preview = d["text"].replace("\n", " ")[:max_chars]
        print(f"Chunk {i} | Fonte: {d['source']}\nTrecho: {preview}...\n")

tests = [
    "Quem é Capitu em 'Dom Casmurro' e como ela é descrita?",
    "Qual a relação entre Bentinho e Escobar em 'Dom Casmurro'?",
    "Como o narrador (Bentinho) apresenta suas memórias e qual o tom predominante da narrativa?",
]


for q in tests:
    print("\n=== Pergunta (PT-BR) ===\n", q)

    # Baseline (sem RAG)
    base = baseline_generate_node({"query": q})
    print("\n— Sem RAG —\n", base.get("resposta_baseline", ""))

    # RAG
    rag = app.invoke({"query": q})
    print("\n— Com RAG —\n", rag.get("resposta_rag", ""))
    fontes = rag.get("fontes", [])
    docs_usados = rag.get("docs", [])
    if not docs_usados:
        docs_usados = retrieve_node({"query": q}).get("docs", [])
    print("\nFontes usadas:", fontes or list({d['source'] for d in docs_usados}))
    print("\nChunks (pré-visualização):\n")
    show_chunks(docs_usados)

  +-----------+  
  | __start__ |  
  +-----------+  
        *        
        *        
        *        
  +----------+   
  | retrieve |   
  +----------+   
        *        
        *        
        *        
+--------------+ 
| generate_rag | 
+--------------+ 
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    

=== Pergunta (PT-BR) ===
 Quem é Capitu em 'Dom Casmurro' e como ela é descrita?

— Sem RAG —
 Capitu é a protagonista feminina do romance "Dom Casmurro", de Machado de Assis. Ela é descrita como uma mulher de olhos "devassos" e "obscuros", com um olhar enigmático que provoca ambiguidade. Sua personalidade é marcada pela inteligência, astúcia e um certo mistério, o que leva o narrador, Bentinho, a questionar sua fidelidade ao longo da história.

— Com RAG —
 Capitu é uma personagem de "Dom Casmurro" descrita como uma figura angelical, sendo comparada a um anjo e considerada a "flôr da casa" e o "sol das manhãs". E

## 7) Conclusão
- Em PT‑BR, a diferença entre **Sem RAG** e **Com RAG** fica nítida, pois a resposta com RAG é **ancorada** nos **chunks** recuperados de Machado de Assis (Project Gutenberg).
- Para produção, substitua FAISS local por um **vector store** persistente e adicione **citações** mais formais (por ex., offsets/IDs dos chunks).